# Day 2 lab, Part C: the dataset

**Hardware status:** not tested on hardware (prepared on 2026-10-02). This
notebook needs no board. It ran on a laptop CPU with the fallback dataset.

**Goal.** You check your recordings, split them by recording session, and
measure the effect of a wrong split.

**Deliverable.** The file `split.json` in your data folder, and the data card
in `report.md`.

**Time.** This notebook: 15 min of the 40 min of Part C. The complete
notebook runs in less than one minute.

**How to work.** Run the cells in order. A cell with the mark
`TODO (student)` has a task. The notebook has five tasks. The cell after a
task checks your result.

## 0. Setup

Run this cell first. It prints the versions that you write in your report,
and it selects the data folder.

- If the folder `data/` of the lab has recordings, the notebook uses them.
- If not, the notebook makes the fallback dataset in `fallback_data/`. The
  fallback signals are simulated. Nobody recorded them.

In [ ]:
import hashlib
import json
import os
import platform
import sys

import matplotlib.pyplot as plt
import numpy as np

print("python    ", sys.version.split()[0])
print("numpy     ", np.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("machine   ", platform.machine())

RATE_HZ = 50            # nominal sampling rate of the lab
ACCEL_RANGE_G = 16      # range of the accelerometer
CLASSES = ["idle", "terrestrial", "lift", "maritime"]
AXES = ["ax", "ay", "az", "gx", "gy", "gz"]


def recording_files(folder):
    '''Return the names of the recording files in a folder.'''
    if not os.path.isdir(folder):
        return []
    return sorted(name for name in os.listdir(folder)
                  if name.endswith(".csv") and len(name.split(".")) == 4)


# The lab folder. The notebook of the folder solutions/ uses the folder above.
LAB = "." if os.path.exists(os.path.join("host", "motion_sim.py")) else ".."

DATA = os.path.join(LAB, "data")
if not recording_files(DATA):
    DATA = os.path.join(LAB, "fallback_data")
    if not recording_files(DATA):
        sys.path.insert(0, os.path.join(LAB, "host"))
        import make_fallback_dataset
        import motion_sim
        os.makedirs(DATA, exist_ok=True)
        for label in motion_sim.CLASSES:
            for session in make_fallback_dataset.SESSIONS:
                for number in range(1, make_fallback_dataset.RECORDINGS + 1):
                    make_fallback_dataset.write_recording(DATA, label, session,
                                                          number)
    print()
    print("No recording in data/. The notebook uses the fallback dataset.")
    print("The fallback signals are simulated. Nobody recorded them.")
print()
print("data folder:", DATA, "with", len(recording_files(DATA)), "files")

## 1. Load the recordings (2 min)

The logger wrote one file for each recording. The file name gives the label,
the session, and the number: `lift.s2.07.csv`. The lines that start with `#`
hold the settings. The columns are:

| Column | Content | Unit |
|---|---|---|
| `n` | sample number, starts at 0 | none |
| `t_us` | time of the board since the first sample | microseconds |
| `ax`, `ay`, `az` | acceleration | $g$ |
| `gx`, `gy`, `gz` | angular rate | degrees each second |

You do not change this code.

In [ ]:
def load_recording(folder, name):
    '''Return one recording as a dictionary.'''
    label, session, number, _ = name.split(".")
    meta = {}
    rows = []
    with open(os.path.join(folder, name), encoding="ascii",
              errors="replace") as handle:
        for line in handle:
            line = line.strip()
            if line.startswith("#"):
                key, _, value = line[1:].partition(":")
                meta[key.strip()] = value.strip()
                continue
            fields = line.split(",")
            if len(fields) != 8 or fields[0] == "n":
                continue
            rows.append([float(f) for f in fields])
    table = np.array(rows)
    return {
        "name": name,
        "label": label,
        "session": session,
        "number": int(number),
        "meta": meta,
        "n": table[:, 0].astype(int),
        "t_us": table[:, 1],
        "data": table[:, 2:8],       # ax, ay, az, gx, gy, gz
    }


records = [load_recording(DATA, name) for name in recording_files(DATA)]
labels = [c for c in CLASSES if any(r["label"] == c for r in records)]
labels += sorted({r["label"] for r in records} - set(labels))
sessions = sorted({r["session"] for r in records})

print(f"{len(records)} recordings, {len(labels)} classes, "
      f"{len(sessions)} sessions")
print()
print(f"{'recordings':<14}" + "".join(f"{s:>6}" for s in sessions) + "   total")
for label in labels:
    counts = [sum(1 for r in records
                  if r["label"] == label and r["session"] == s)
              for s in sessions]
    print(f"{label:<14}" + "".join(f"{c:>6}" for c in counts)
          + f"{sum(counts):>8}")

## 2. Look at one recording of each class (2 min)

The plot shows the first recording of each class: the three acceleration
axes. Compare each plot with the definition of the class:

- **idle:** no motion. The z axis shows gravity, about 1 $g$.
- **terrestrial:** motion in the horizontal plane.
- **lift:** motion up and down, on the z axis.
- **maritime:** slow motion on all axes.

In [ ]:
figure, axes = plt.subplots(1, len(labels), figsize=(3.6 * len(labels), 2.8),
                            sharey=True)
for axis, label in zip(np.atleast_1d(axes), labels):
    record = next(r for r in records if r["label"] == label)
    seconds = record["t_us"] / 1e6
    for k in range(3):
        axis.plot(seconds, record["data"][:, k], label=AXES[k], linewidth=0.9)
    axis.set_title(f"{label} ({record['name']})", fontsize=9)
    axis.set_xlabel("time (s)")
    axis.grid(alpha=0.3)
np.atleast_1d(axes)[0].set_ylabel("acceleration (g)")
np.atleast_1d(axes)[-1].legend(loc="upper right", fontsize=7, ncol=3)
figure.tight_layout()
plt.show()

## 3. Quality checks (4 min)

Part 3 of the lecture gave two types of check.

**Structure checks.** Does the file have the correct form? The code below
counts the rows and looks for a gap in the sample number.

**Statistical checks.** Do the values make sense? The code below looks for
values at the limit of the range, and you calculate the real sampling rate.

**Task 1.** Write the function `mean_rate`. It returns the mean sampling
rate of one recording in Hz. Use the time of the board, `t_us`, in
microseconds. The rate is the number of periods divided by the time between
the first sample and the last sample.

In [ ]:
def mean_rate(t_us):
    '''Return the mean sampling rate in Hz from the time stamps in microseconds.'''
    # TODO (student): replace the line below with your calculation.
    return 0.0

In [ ]:
def check_mean_rate():
    '''Compare mean_rate with two known cases.'''
    exact = np.arange(500) * 20000.0            # 50 Hz, 500 samples
    slow = np.arange(100) * 24000.0             # a period of 24 ms: 41.67 Hz
    correct = (abs(mean_rate(exact) - 50.0) < 1e-6
               and abs(mean_rate(slow) - 1e6 / 24000.0) < 1e-6)
    print("Task 1:", "complete" if correct else "not complete")
    return correct


task_1_complete = check_mean_rate()

In [ ]:
def check_recording(record, seconds=10.0):
    '''Return the checks of one recording as a dictionary.'''
    n = record["n"]
    accel = record["data"][:, 0:3]
    length = np.sqrt((accel ** 2).sum(axis=1))
    lost = int((n[-1] - n[0] + 1) - len(n))
    rate = mean_rate(record["t_us"])
    clipped = int((np.abs(accel).max(axis=1) >= 0.99 * ACCEL_RANGE_G).sum())
    problems = []
    if len(n) != int(seconds * RATE_HZ):
        problems.append("rows")
    if lost:
        problems.append("lost samples")
    if abs(rate - RATE_HZ) > 0.01 * RATE_HZ:
        problems.append("rate")
    if clipped:
        problems.append("clipping")
    if record["label"] == "idle" and abs(length.mean() - 1.0) > 0.05:
        problems.append("gravity is not 1 g")
    if record["label"] != "idle" and accel.std(axis=0).max() < 0.02:
        problems.append("no motion")
    return {"rows": len(n), "lost": lost, "rate": rate, "clipped": clipped,
            "mean_length": length.mean(), "problems": problems}


checks = {r["name"]: check_recording(r) for r in records}
bad = {name: c for name, c in checks.items() if c["problems"]}

rates = np.array([c["rate"] for c in checks.values()])
print(f"recordings checked        {len(checks)}")
print(f"rows in one recording     {min(c['rows'] for c in checks.values())} to "
      f"{max(c['rows'] for c in checks.values())}")
print(f"mean rate, all recordings {rates.mean():.2f} Hz "
      f"(minimum {rates.min():.2f}, maximum {rates.max():.2f})")
print(f"lost samples              {sum(c['lost'] for c in checks.values())}")
print(f"clipped samples           {sum(c['clipped'] for c in checks.values())}")
print(f"recordings with a problem {len(bad)}")
for name, c in list(bad.items())[:5]:
    print(f"  {name}: {', '.join(c['problems'])}")
if len(bad) > 5:
    print(f"  and {len(bad) - 5} more")
if not task_1_complete:
    print()
    print("Task 1 is not complete, so each recording has the problem 'rate'.")

A recording with a problem must not go into the dataset. Delete the file and
record it again. Then run the notebook again from section 0.

The next cell gives the duration of each class. You need these numbers for
the data card.

In [ ]:
print(f"{'class':<14}{'recordings':>12}{'seconds':>10}")
total = 0.0
for label in labels:
    group = [r for r in records if r["label"] == label]
    seconds = sum(len(r["n"]) for r in group) / RATE_HZ
    total += seconds
    print(f"{label:<14}{len(group):>12}{seconds:>10.0f}")
print(f"{'total':<14}{len(records):>12}{total:>10.0f}")

## 4. Split by recording session (3 min)

All recordings of one session go to the same set. The check of the lab is:
no session is in the training set and also in the test set.

**Task 2.** Write the function `split_by_session`. It gets the list of the
recordings and the list of the test sessions. It returns two lists: the
training recordings and the test recordings. The session of a recording is
`record["session"]`.

In [ ]:
TEST_SESSIONS = [sessions[-1]]     # the last session is the test session


def split_by_session(records, test_sessions):
    '''Return (training recordings, test recordings).'''
    # TODO (student): replace the two lines below. Put a recording in the
    # test list if its session is one of the test sessions.
    training = list(records)
    test = []
    return training, test

In [ ]:
def check_split(training, test):
    '''Check the rule of the lab for one split.'''
    train_sessions = {r["session"] for r in training}
    test_sessions = {r["session"] for r in test}
    shared = train_sessions & test_sessions
    all_classes = (len(test) > 0 and len(training) > 0
                   and {r["label"] for r in test} == {r["label"] for r in training})
    complete = (len(training) + len(test) == len(records)
                and not shared and all_classes)
    print(f"training: {len(training)} recordings, sessions {sorted(train_sessions)}")
    print(f"test:     {len(test)} recordings, sessions {sorted(test_sessions)}")
    print(f"sessions in the two sets: {sorted(shared) if shared else 'none'}")
    print(f"each class is in the two sets: {'yes' if all_classes else 'no'}")
    if len(sessions) < 2:
        print("The dataset has only one session. A split by session needs a "
              "minimum of two sessions.")
    return complete


train_records, test_records = split_by_session(records, TEST_SESSIONS)
task_2_complete = check_split(train_records, test_records)
print("Task 2:", "complete" if task_2_complete else "not complete")

## 5. Windows (2 min)

The model of Day 3 takes a window of 2 s. A window of 2 s at 50 Hz holds 100
samples. The stride is 0.2 s: 10 samples. The lecture gave the number of
windows of one recording:

$$\text{windows} = \left\lfloor \frac{L - W}{S} \right\rfloor + 1$$

$L$ is the length of the recording, $W$ is the length of the window, and $S$
is the stride.

**Task 3.** Write the function `count_windows`. All three arguments are
numbers of samples.

In [ ]:
WINDOW = 100     # samples: 2 s at 50 Hz
STRIDE = 10      # samples: 0.2 s at 50 Hz


def count_windows(length, window, stride):
    '''Return the number of windows of one recording.'''
    # TODO (student): replace the line below with the formula.
    return 0

In [ ]:
def cut_windows(record, window=WINDOW, stride=STRIDE):
    '''Return the windows of one recording: array (windows, samples, 3 axes).

    The windows hold the three acceleration axes, as in the kit lab.
    '''
    accel = record["data"][:, 0:3]
    starts = range(0, len(accel) - window + 1, stride)
    return np.stack([accel[s:s + window] for s in starts])


example = records[0]
real = len(cut_windows(example))
yours = count_windows(len(example["n"]), WINDOW, STRIDE)
task_3_complete = (yours == real and count_windows(3000, 100, 10) == 291
                   and count_windows(500, 100, 100) == 5)
print(f"{example['name']}: {len(example['n'])} samples give {real} windows. "
      f"Your function gives {yours}.")
print(f"one window holds {WINDOW} x 3 = {WINDOW * 3} values")
print("Task 3:", "complete" if task_3_complete else "not complete")

## 6. The leakage experiment (2 min)

You now measure the effect of a wrong split. The two procedures use the same
recordings and the same classifier.

| | Procedure A (wrong) | Procedure B (correct) |
|---|---|---|
| Step 1 | Cut all recordings into windows | Split the recordings by session |
| Step 2 | Normalize with the statistics of all windows | Cut the windows in each set |
| Step 3 | Mix the windows: 80 percent training, 20 percent test | Normalize with the statistics of the training set |

**The classifier.** For each window, the code calculates six features: the
mean and the standard deviation of each acceleration axis. A new window gets
the class of the nearest training window (nearest neighbour). This is a
simple classifier with no training. Day 3 trains a neural network.

Procedure B runs one time for each session: that session is the test
session, and the other sessions are the training set.

**Task 4.** Write your prediction **before** you run the experiment. The
prediction is part of the assessment. A wrong prediction costs no points. A
missing prediction does.

In [ ]:
# TODO (student): replace each "?".
prediction = {
    # "A", "B", or "equal"
    "the procedure with the higher test accuracy": "?",
    # a number between 0 and 100
    "test accuracy of procedure A in percent": "?",
    # a number between 0 and 100
    "lowest test accuracy of procedure B in percent": "?",
}
for question, answer in prediction.items():
    print(f"{question:<52} {answer}")

In [ ]:
def features(windows):
    '''Return six features for each window: mean and std of each axis.'''
    return np.concatenate([windows.mean(axis=1), windows.std(axis=1)], axis=1)


def windows_and_labels(group):
    '''Return (features, class numbers) of all windows of a list of recordings.'''
    x = np.concatenate([features(cut_windows(r)) for r in group])
    y = np.concatenate([[labels.index(r["label"])] * len(cut_windows(r))
                        for r in group])
    return x, y


def nearest_neighbour(x_train, y_train, x_test):
    '''Return the class of the nearest training window for each test window.'''
    distance = ((x_test[:, None, :] - x_train[None, :, :]) ** 2).sum(axis=2)
    return y_train[distance.argmin(axis=1)]


# Procedure A (wrong): windows first, statistics of all windows, random split.
x_all, y_all = windows_and_labels(records)
x_norm = (x_all - x_all.mean(axis=0)) / np.maximum(x_all.std(axis=0), 1e-9)
order = np.random.default_rng(0).permutation(len(x_all))
cut = int(0.8 * len(x_all))
train_index, test_index = order[:cut], order[cut:]
predicted = nearest_neighbour(x_norm[train_index], y_all[train_index],
                              x_norm[test_index])
accuracy_a = (predicted == y_all[test_index]).mean()

# Procedure B (correct): one run for each test session.
accuracy_b = {}
confusion_b = {}
for session in sessions:
    train_group, test_group = [r for r in records if r["session"] != session], \
                              [r for r in records if r["session"] == session]
    if not train_group or not test_group:
        continue
    x_train, y_train = windows_and_labels(train_group)
    x_test, y_test = windows_and_labels(test_group)
    mean = x_train.mean(axis=0)
    std = np.maximum(x_train.std(axis=0), 1e-9)
    predicted = nearest_neighbour((x_train - mean) / std, y_train,
                                  (x_test - mean) / std)
    accuracy_b[session] = (predicted == y_test).mean()
    matrix = np.zeros((len(labels), len(labels)), dtype=int)
    for true, guess in zip(y_test, predicted):
        matrix[true, guess] += 1
    confusion_b[session] = matrix

print(f"windows in total: {len(x_all)}")
print()
print(f"Procedure A, random split of the windows: {100 * accuracy_a:6.1f} percent")
for session, accuracy in accuracy_b.items():
    print(f"Procedure B, test session {session:<4}            "
          f"{100 * accuracy:6.1f} percent")
if accuracy_b:
    print(f"Procedure B, mean of the sessions:        "
          f"{100 * np.mean(list(accuracy_b.values())):6.1f} percent")

The total accuracy hides which class fails. The next cell shows the
confusion matrix of procedure B for the session with the lowest accuracy.
Each row is the real class. Each column is the class that the classifier
gave.

In [ ]:
if accuracy_b:
    worst = min(accuracy_b, key=accuracy_b.get)
    print(f"Procedure B, test session {worst}: "
          f"{100 * accuracy_b[worst]:.1f} percent")
    print()
    print(f"{'real class':<14}" + "".join(f"{label[:6]:>8}" for label in labels))
    for label, row in zip(labels, confusion_b[worst]):
        print(f"{label:<14}" + "".join(f"{value:>8}" for value in row))
else:
    print("The dataset has only one session. Procedure B is not possible.")

> TODO (student): answer these three questions in `report.md`.
>
> 1. Compare the result with your prediction of task 4.
> 2. Why is the result of procedure A too high? Use the words "overlap" and
>    "session".
> 3. The results of procedure B are different for each test session. Which
>    number do you give to a customer, and why?

## 7. Save the split and the data card numbers (2 min)

The next cell writes `split.json` into your data folder. The file holds the
version of the dataset, the sessions of each set, the file names, and a hash
value of each file. Day 3 reads this file. The program
`host/to_edge_impulse.py` also reads it.

A hash value changes when one byte of the file changes. It shows later if a
file is the same file.

In [ ]:
def sha256_of(path):
    '''Return the SHA-256 hash value of a file as text.'''
    with open(path, "rb") as handle:
        return hashlib.sha256(handle.read()).hexdigest()


split = {
    "dataset": "motion of a package",
    "version": "v1",
    "rate_hz": RATE_HZ,
    "window_samples": WINDOW,
    "stride_samples": STRIDE,
    "classes": labels,
    "training_sessions": sorted({r["session"] for r in train_records}),
    "test_sessions": sorted({r["session"] for r in test_records}),
    "training": [r["name"] for r in train_records],
    "test": [r["name"] for r in test_records],
    "sha256": {r["name"]: sha256_of(os.path.join(DATA, r["name"]))
               for r in records},
}

if task_2_complete:
    with open(os.path.join(DATA, "split.json"), "w", encoding="utf-8") as handle:
        json.dump(split, handle, indent=2)
    print("wrote", os.path.join(DATA, "split.json"))
else:
    print("Task 2 is not complete. The notebook did not write split.json.")

print()
print("Numbers for the data card:")
print(f"  classes:            {', '.join(labels)}")
print(f"  sampling rate:      {rates.mean():.2f} Hz (nominal {RATE_HZ} Hz)")
for label in labels:
    group = [r for r in records if r["label"] == label]
    print(f"  duration, {label:<12} {sum(len(r['n']) for r in group) / RATE_HZ:.0f} s")
print(f"  sessions:           {', '.join(sessions)}")
print(f"  training sessions:  {', '.join(split['training_sessions'])}")
print(f"  test sessions:      {', '.join(split['test_sessions']) or 'none'}")
print(f"  recordings:         {len(split['training'])} training, "
      f"{len(split['test'])} test")
print(f"  windows:            "
      f"{sum(count_windows(len(r['n']), WINDOW, STRIDE) for r in train_records)} training, "
      f"{sum(count_windows(len(r['n']), WINDOW, STRIDE) for r in test_records)} test")

**Task 5.** Write the data card in `report.md`. Use the numbers of the cell
above. Write also the known limits of your dataset: where can a model that
learns from this data fail?

> TODO (student): write the data card in `report.md`.

## Summary

- A recording is one file with one label and one session.
- The structure check and the statistical check find a wrong recording
  before the training.
- A split of the windows at random gives a test accuracy that is too high.
  A split by session gives the accuracy for a new session.
- `split.json` and the data card make the dataset a version that Day 3 can
  use.